# Projeto Final – Storytelling sobre Desastres Naturais (2014—2021)

## Setup

### Imports

In [52]:
import os
import pandas as pd
import plotly.express as px
from typing import List
from dotenv import load_dotenv
from mysql.connector import connect

### Environment Variables

In [53]:
load_dotenv()

MYSQL_HOST=os.getenv(key="MYSQL_HOST")
MYSQL_PORT=os.getenv(key="MYSQL_PORT")
MYSQL_USER=os.getenv(key="MYSQL_USER")
MYSQL_PASS=os.getenv(key="MYSQL_PASS")
MYSQL_DB=os.getenv(key="MYSQL_DB")

### MySQL Connector

In [134]:
connector = connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASS,
    database=MYSQL_DB,
    ssl_disabled=False
)

### Dictionaries

In [84]:
brazil_states: dict[str, str] = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}

## Plot Charts

Q1 – _Quais tipos de desastre predominaram entre 2014 e 2021?_

In [107]:
q1_query = """
    SELECT
        cobrade AS desastre,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY cobrade
    ORDER BY ocorrencias DESC
    LIMIT 10;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q1_query)
rows = cursor.fetchall()


q1_items: List[dict] = []
for row in rows:
    temp: List[str] = row.get("desastre", "").split('-')  # type: ignore (Pylance)

    if len(temp) > 2 and ',' not in temp[2]:
        disaster = temp[2]
    else:
        disaster = temp[1]

    q1_items.append({
        "disaster": disaster,
        "occurrences": row.get("ocorrencias", "")  # type: ignore (Pylance)
    })


q1_df = pd.DataFrame(q1_items)

Criar gráfico
- Barras (horizontal)

In [87]:
fig = px.bar(
    q1_df.sort_values("occurrences"),
    x="occurrences",
    y="disaster",
    orientation="h",
    text="occurrences",
    color="occurrences",
    color_continuous_scale="Teal"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    title="Desastres Predominantes entre 2014 e 2021",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Nº de Casos",
    yaxis_title="",
    showlegend=False,
    coloraxis_colorbar=dict(title="")
)

fig.show()

Q2 – _Onde os desastres se concentraram geograficamente?_

In [105]:
q2_query = """
    SELECT
        uf,
        municipio,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY uf, municipio
    ORDER BY ocorrencias DESC
    LIMIT 20;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q2_query)
rows = cursor.fetchall()


q2_items = [
    {
        "state": row.get("uf", ""),                 # type: ignore (Pylance)
        "city": row.get("municipio", ""),           # type: ignore (Pylance)
        "occurrences": row.get("ocorrencias", "")   # type: ignore (Pylance)
    }
    for row in rows
]


q2_df = pd.DataFrame(q2_items)

Criar gráfico
- Treemap

In [106]:
q2_df["full_state"] = q2_df["state"].map(brazil_states)

fig = px.treemap(
    q2_df,
    path=["full_state", "city"],
    values="occurrences",
    color="occurrences",
    color_continuous_scale="tempo"
)

fig.update_traces(
    texttemplate="%{label}<br>%{value}"
)

fig.update_layout(
    title="Maior Concentração Geográfica de Desastres",
    title_x=0.5,
    template="plotly_white",
    coloraxis_colorbar=dict(title="")
)

fig.show()

Q3 – _Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**antes do COVID**)_

In [ ]:
q3_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido' AND ano BETWEEN 2014 and 2019 -- sem COVID
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 5;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_query)
rows = cursor.fetchall()


q3_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_df = pd.DataFrame(q3_items)

Criar gráfico
- Bolha

In [133]:
q3_df["location"] = q3_df["city"] + " (" + q3_df["state"] + ")"
q3_df["deaths"] = pd.to_numeric(q3_df["deaths"], errors="coerce")

fig = px.scatter(
    q3_df,
    x="location",
    y="disaster",
    size="deaths",
    color="deaths",
    size_max=80,
    color_continuous_scale="burg"
)

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=2.0 * q3_df["deaths"].max() / (80 ** 2),
        sizemin=25,
        line=dict(width=2, color="black")
    )
)

for _, row in q3_df.iterrows():
    font_color = "white" if row["deaths"] >= 100 else "black"
    fig.add_annotation(
        x=row["location"],
        y=row["disaster"],
        text=f"{int(row['deaths'])}",
        showarrow=False,
        font=dict(size=12, color=font_color)
    )

fig.update_layout(
    title="Desastres com Maior Número de Mortes (antes do COVID)",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Cidade (UF)",
    yaxis_title="",
    coloraxis_colorbar=dict(title="")
)

fig.show()

Q3 – Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**depois do COVID**)

In [137]:
q3_covid_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido'
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 5;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_covid_query)
rows = cursor.fetchall()


q3_covid_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_covid_df = pd.DataFrame(q3_covid_items)

Criar gráfico
- Bubble

In [139]:
q3_covid_df["location"] = q3_covid_df["city"] + " (" + q3_covid_df["state"] + ")"
q3_covid_df["deaths"] = pd.to_numeric(q3_covid_df["deaths"], errors="coerce")

fig = px.scatter(
    q3_covid_df,
    x="location",
    y="disaster",
    size="deaths",
    color="deaths",
    size_max=80,
    color_continuous_scale="burg"
)

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=2.0 * q3_covid_df["deaths"].max() / (80 ** 2),
        sizemin=25,
        line=dict(width=2, color="black")
    )
)

for _, row in q3_covid_df.iterrows():
    font_color = "white" if row["deaths"] >= 30000 else "black"
    fig.add_annotation(
        x=row["location"],
        y=row["disaster"],
        text=f"{int(row['deaths'])}",
        showarrow=False,
        font=dict(size=12, color=font_color)
    )

fig.update_layout(
    title="Desastres com Maior Número de Mortes (depois do COVID)",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Cidade (UF)",
    yaxis_title="",
    coloraxis_colorbar=dict(title="")
)

fig.show()